# 분류 모델 학습 노트북 v2
### 📝 변경 내역
| 버전 | 바뀐 설정 | 결과 (val macro F1) |
| --- | --- | --- |
| v0 | 기본값 (1,000건 샘플, 1 epoch) | 0.0793 |
| v1 | `EPOCHS = 3`, `SAMPLE_SIZE = None` (전체 데이터) | 0.7527 |
| v2 | `MODEL_NAME = "klue/roberta-base"` | 실행 후 기록 |

### 🔁 새 버전 실행하는 법
1. 이전 버전 파일 복사, 파일명 → `notebooks/{a|b}_train_v{번호}.ipynb` (예: `a_train_v1`, `b_train_v3`)

2. 복사본의 **설정 셀만** 수정
- ⚠️ `SAVE_DIR`의 버전 이름은 **반드시** 새 번호로 변경 (안 바꾸면 이전 결과에 덮어씀)
- SAVE_DIR = os.path.join(PROJECT_DIR, "checkpoints", **"v2"**)

4. 이 칸 맨 위 **변경 내역**에 무엇을 바꿨는지 기록 (a_train_v1.ipynb처럼)

### ✅ 실행 전 확인
1. 오른쪽 위 **Select Kernel** → **Python 환경** → **`.venv`** 선택 (`~/project/.venv/bin/python`, Python 3.11)
- 시스템 Python(`/usr/bin/python`, 3.12)은 선택 X
- 패키지는 `.venv`에 설치돼 있음, 따로 설치 X
2. 터미널 `nvidia-smi`로 GPU 비어 있는지 확인 → GPU 1장 공유라 **돌리기 전 카톡**
3. 위에서부터 차례로 실행 (**Run All**)

> 새 패키지가 필요하면: 터미널에서 `source ~/project/.venv/bin/activate` → `uv pip install 이름` → `requirements.txt`에도 한 줄 추가 (`pip install` 직접 금지)

### 🚫 규칙
- 공식 **test는 절대 쓰지 않음** (W9 최종 평가 때 한 번만). 점수 확인·체크포인트 선택은 전부 **validation**으로
- 실행 중 커널 재시작 X (학습 결과 변수가 사라짐. 저장된 파일은 남음)
- 결과는 `checkpoints/v{번호}/final/`의 `val_result.json`(점수), `train_config.json`(설정)에서 확인


## 0. 설정 (실험할 때 여기만 바꾸기)

In [1]:
# ===== 설정 (실험할 때 여기만 바꾸기) =====
import os

os.environ["HF_HOME"] = "/nfs/team/hf_cache"

MODEL_NAME = "klue/roberta-base"   # v2: KcELECTRA 대신 KLUE-RoBERTa 비교
MAX_LENGTH = 64         # B 기록: 토큰 길이 95% 지점 37 → 32의 배수로 올려 64
BATCH_SIZE = 32
LR = 2e-5
EPOCHS = 3              # v1과 동일하게 유지
SEED = 42
SAMPLE_SIZE = None      # v0: 1000, v1: None (전체 train 78,305건)
THRESHOLD = 0.5         # W6 B가 validation으로 조정
USE_CLASS_WEIGHT = False  # W5 B 불균형 처리 실험 (True면 드문 라벨에 loss 가중치)

# --- 경로 (서버) ---
PROJECT_DIR = os.path.realpath(os.path.expanduser("~/project"))   # data/, models/ 폴더가 있는 곳
SAVE_DIR = os.path.join(PROJECT_DIR, "checkpoints", "v2")         # v2 체크포인트 저장

# --- 거의 안 바꾸는 값 ---
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

print("PROJECT_DIR:", PROJECT_DIR)
print("SAVE_DIR:", SAVE_DIR)

PROJECT_DIR: /home/team17/project
SAVE_DIR: /home/team17/project/checkpoints/v2


## 1. 환경 준비

- 패키지 설치: **처음 한 번만** 실행하고, 끝나면 위쪽 **Restart**로 커널을 다시 시작한 뒤 다음 셀부터 실행
- 서버에 이미 설치돼 있으면 건너뛰어도 됨 (아래 확인 셀에서 torch·GPU가 보이면 OK)
- B 파일 경로 확인 + `__init__.py` 자동 생성

In [2]:
import os, sys

os.makedirs(SAVE_DIR, exist_ok=True)

# B 파일 import 준비
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for pkg in ["data", "models"]:
    os.makedirs(pkg, exist_ok=True)
    open(os.path.join(pkg, "__init__.py"), "a").close()   # import되려면 필요

missing = [f for f in ["data/kmhas.py", "models/metrics.py"] if not os.path.exists(f)]
assert not missing, f"B 파일이 없어요: {missing} → VS Code 탐색기에서 {PROJECT_DIR} 아래에 만들기"
print("B 파일 확인 OK / 체크포인트 저장 위치:", SAVE_DIR)

B 파일 확인 OK / 체크포인트 저장 위치: /home/team17/project/checkpoints/v2


## 2. 데이터 불러오기 (B의 `load_kmhas`)

In [3]:
import random
import numpy as np
import torch
from transformers import set_seed

from data.kmhas import load_kmhas, LABEL_NAMES
from models.metrics import evaluate

set_seed(SEED)   # random · numpy · torch 시드를 한 번에 고정
NUM_LABELS = len(LABEL_NAMES)   # 9
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "-")
if device == "cpu":
    print("⚠️ GPU가 안 잡혀요. 커널이 서버 Python인지, nvidia-smi에 GPU가 보이는지 확인")

train_df, val_df, test_df = load_kmhas()   # B 기록: train 78305 / val 8776 / test 21939
del test_df   # test는 W9 최종 평가 때만. 실수로 쓰지 않도록 바로 지움

if SAMPLE_SIZE:
    train_df = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), random_state=SEED).reset_index(drop=True)
print(f"학습에 쓰는 데이터: train {len(train_df)} / val {len(val_df)}")
train_df.head()

/home/team17/project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda | GPU: NVIDIA RTX A5000
[train] 빈 문장 -0, 중복 -421, test와 겹침 -251 → 78305건
[최종] train 78305 / val 8776 / test 21939
학습에 쓰는 데이터: train 78305 / val 8776


,text,labels
0,"""자한당틀딱들.. 악플질 고만해라.""","[0, 0, 1, 0, 1, 0, 0, 0, 0]"
1,정치적으로 편향된 평론한은 분은 별로...,"[0, 0, 0, 0, 0, 0, 0, 0, 1]"
2,적당히좀 쳐먹지.그랬냐??? 안그래도 문재인 때문에 나라 엉망진창인데...,"[0, 0, 1, 0, 0, 0, 0, 0, 0]"
3,"""안서는 아재들 풀발기 ㅋㄲㅋ""","[0, 0, 0, 0, 1, 0, 0, 0, 0]"
4,우와 ㅋ 능력자,"[0, 0, 0, 0, 0, 0, 0, 0, 1]"


## 3. ① 토크나이징 + 학습용 Dataset 변환

- 문장 → `input_ids`, `attention_mask` (길이 `MAX_LENGTH`에서 자름)
- KcELECTRA는 바이트 단위 BPE 토크나이저라 토큰이 `'Ġì...'`처럼 깨진 글자로 보임 → 정상
- 패딩은 배치마다 가장 긴 문장에 맞춤 (`DataCollatorWithPadding`) → 짧은 댓글이 많아서 더 빠름
- 다중 라벨은 BCE loss를 쓰므로 정답 라벨을 **float**로 바꿔야 함 (`[0,0,1,1,0,...]` → `[0.,0.,1.,1.,0.,...]`)

In [4]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# 저장소의 토크나이저를 그대로 씀. KcELECTRA는 바이트 단위 BPE라
# 토큰이 'Ġì...'처럼 깨진 글자로 보이는 게 정상 (한글을 바이트로 쪼갠 표기)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 토크나이저 점검: [UNK] 비율이 높으면 설정이 잘못된 것 (W5에 모델을 바꿀 때도 확인용)
_chk = tokenizer(train_df["text"][:200].tolist())["input_ids"]
_unk = sum(i == tokenizer.unk_token_id for ids in _chk for i in ids) / sum(len(ids) for ids in _chk)
print(f"토크나이저: {type(tokenizer).__name__} | 단어 수 {len(tokenizer)} | [UNK] 비율 {_unk:.1%}")
assert _unk < 0.05, "[UNK]가 너무 많아요 → 토크나이저 설정 확인"

def to_dataset(df):
    ds = Dataset.from_dict({
        "text": df["text"].tolist(),
        "labels": [[float(x) for x in l] for l in df["labels"]],
    })
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH),
                batched=True, remove_columns=["text"])
    return ds

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 확인: 첫 문장이 어떻게 바뀌었는지
ex = train_ds[0]
print(train_df["text"][0])
print(tokenizer.convert_ids_to_tokens(ex["input_ids"])[:20])
print("labels:", ex["labels"])

토크나이저: BertTokenizer | 단어 수 32000 | [UNK] 비율 0.5%


Map: 100%|██████████| 8776/8776 [00:00<00:00, 20263.51 examples/s]

"자한당틀딱들.. 악플질 고만해라."
['[CLS]', '"', '자', '##한', '##당', '##틀', '##딱', '##들', '.', '.', '악', '##플', '##질', '고만', '##해', '##라', '.', '"', '[SEP]']
labels: [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]


## 4. ② KcELECTRA를 9개 라벨 다중 분류로 불러오기

- `problem_type="multi_label_classification"` → 출력 9개 각각에 sigmoid + BCE loss (한 문장에 라벨 여러 개 가능)
- `id2label`을 넣어두면 저장된 모델만 불러와도 라벨 이름이 따라옴 (W4 `predict()`, 데모에서 편함)
- `classifier ... newly initialized` 경고는 정상 (분류층은 새로 학습하는 부분)

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label={i: n for i, n in enumerate(LABEL_NAMES)},
    label2id={n: i for i, n in enumerate(LABEL_NAMES)},
)
print(model.config.problem_type, model.config.num_labels)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5125.00it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


multi_label_classification 9


## 5. ④ 학습 중 점수 계산에 B의 `evaluate()` 연결 + (선택) 클래스 가중치

- `compute_metrics`: 모델 출력(logit) → sigmoid 확률 → B의 `evaluate(y_true, y_prob, threshold)` → micro/macro F1
- 결과표 형식이 B 함수 하나로 통일돼서 W4 v1, W5 v2와 바로 비교 가능
- `USE_CLASS_WEIGHT=True`면 라벨별 `pos_weight = 음성 수 / 양성 수`로 드문 라벨(예: race 237건)을 더 크게 반영 (W5 B 실험용, v0는 False)

In [6]:
from transformers import Trainer

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    r = evaluate(labels, sigmoid(logits), threshold=THRESHOLD, show=False)
    out = {"micro_f1": r["micro_f1"], "macro_f1": r["macro_f1"]}
    out.update({f"f1_{k}": v for k, v in r["per_label_f1"].items()})
    return out

# 클래스 가중치 (USE_CLASS_WEIGHT=True일 때만 사용)
y_train = np.array(train_df["labels"].tolist())
pos = y_train.sum(axis=0)
pos_weight = torch.tensor((len(y_train) - pos) / np.maximum(pos, 1), dtype=torch.float)

class WeightedTrainer(Trainer):
    """BCE loss에 라벨별 pos_weight를 거는 Trainer"""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

print("train 라벨별 개수:", dict(zip(LABEL_NAMES, pos.astype(int))))
if USE_CLASS_WEIGHT:
    print("pos_weight:", dict(zip(LABEL_NAMES, pos_weight.numpy().round(1))))

train 라벨별 개수: {'origin': np.int64(7194), 'physical': np.int64(6229), 'politics': np.int64(8436), 'profanity': np.int64(11547), 'age': np.int64(5185), 'gender': np.int64(5680), 'race': np.int64(237), 'religion': np.int64(1738), 'not_hate': np.int64(42845)}


## 6. ③ Trainer 설정 · 학습 · 체크포인트 저장

- epoch마다 validation 평가 + 체크포인트를 `SAVE_DIR`(드라이브)에 저장
- 끝나면 **validation macro F1이 가장 좋은 체크포인트**를 자동으로 다시 불러옴 (불균형 데이터라 micro보다 macro 기준)
- 체크포인트는 최근 2개만 남김 (드라이브 용량 절약)

In [7]:
import math
from transformers import TrainingArguments

# warmup은 '단계 수(정수)'로 직접 계산 → transformers v4·v5 어느 버전이든 동작 (v5에서 warmup_ratio 인자가 없어짐)
total_steps = math.ceil(len(train_ds) / BATCH_SIZE) * EPOCHS
warmup_steps = math.ceil(total_steps * WARMUP_RATIO)
print(f"총 학습 step {total_steps} / warmup {warmup_steps}")

args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

TrainerClass = WeightedTrainer if USE_CLASS_WEIGHT else Trainer
trainer = TrainerClass(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

총 학습 step 7344 / warmup 735


In [9]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,F1 Origin,F1 Physical,F1 Politics,F1 Profanity,F1 Age,F1 Gender,F1 Race,F1 Religion,F1 Not Hate
1,0.106456,0.109273,0.838637,0.703076,0.724299,0.770381,0.814118,0.834049,0.762097,0.721440,0.000000,0.807818,0.893480
2,0.090787,0.099802,0.849780,0.723547,0.784367,0.786982,0.827548,0.852804,0.790076,0.746091,0.000000,0.828571,0.895482
3,0.063999,0.100216,0.853425,0.725750,0.784793,0.794346,0.832009,0.849980,0.809651,0.742768,0.000000,0.818505,0.899693


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.56it/s]


TrainOutput(global_step=7344, training_loss=0.11430182639907129, metrics={'train_runtime': 320.5456, 'train_samples_per_second': 732.86, 'train_steps_per_second': 22.911, 'total_flos': 6910380160481952.0, 'train_loss': 0.11430182639907129, 'epoch': 3.0})


## 7. validation 결과표 (B의 `evaluate`, 라벨별 리포트)

In [10]:
pred = trainer.predict(val_ds)
val_prob = sigmoid(pred.predictions)
result = evaluate(pred.label_ids, val_prob, threshold=THRESHOLD, show=True)

                precision    recall  f1-score   support

    origin(출신)       0.81      0.76      0.78       759
  physical(외모)       0.81      0.78      0.79       717
  politics(정치)       0.85      0.82      0.83       896
 profanity(욕설)       0.86      0.84      0.85      1291
       age(연령)       0.80      0.82      0.81       552
    gender(성별)       0.77      0.72      0.74       659
      race(인종)       0.00      0.00      0.00        24
  religion(종교)       0.82      0.82      0.82       140
not_hate(해당없음)       0.90      0.90      0.90      4887

     micro avg       0.86      0.85      0.85      9925
     macro avg       0.73      0.72      0.73      9925
  weighted avg       0.86      0.85      0.85      9925
   samples avg       0.86      0.86      0.85      9925

micro F1 0.8534 | macro F1 0.7257


## 8. 최종 모델 저장 + 저장 확인

- `SAVE_DIR/final`에 모델·토크나이저·설정값·결과를 함께 저장 → W4 `predict()`는 이 폴더만 불러오면 됨
- 저장한 폴더에서 다시 불러와 예시 문장을 예측해 봄 (서버 디스크에 제대로 저장됐는지 확인) (v0는 1,000건·1 epoch라 대부분 `not_hate`로 나와도 정상)

In [11]:
import json

FINAL_DIR = os.path.join(SAVE_DIR, "final")
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

config = dict(MODEL_NAME=MODEL_NAME, MAX_LENGTH=MAX_LENGTH, BATCH_SIZE=BATCH_SIZE, LR=LR,
              EPOCHS=EPOCHS, SEED=SEED, SAMPLE_SIZE=SAMPLE_SIZE, THRESHOLD=THRESHOLD,
              USE_CLASS_WEIGHT=USE_CLASS_WEIGHT, n_train=len(train_ds), n_val=len(val_ds))
with open(os.path.join(FINAL_DIR, "train_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(os.path.join(FINAL_DIR, "val_result.json"), "w", encoding="utf-8") as f:
    json.dump({k: (float(v) if not isinstance(v, dict) else {kk: float(vv) for kk, vv in v.items()})
               for k, v in result.items()}, f, ensure_ascii=False, indent=2)

print("저장된 체크포인트:", sorted(os.listdir(SAVE_DIR)))
print("final 폴더:", sorted(os.listdir(FINAL_DIR)))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.50it/s]

저장된 체크포인트: ['checkpoint-4896', 'checkpoint-7344', 'final']
final 폴더: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'train_config.json', 'training_args.bin', 'val_result.json']


In [12]:
# 저장한 폴더에서 다시 불러와 예측 (저장이 제대로 됐는지 확인)
tok2 = AutoTokenizer.from_pretrained(FINAL_DIR)
model2 = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(device).eval()

samples = ["오늘 날씨 진짜 좋다", "이 정치인은 정말 최악이다", "나이 먹고 왜 저러냐"]
with torch.no_grad():
    enc = tok2(samples, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(device)
    probs = torch.sigmoid(model2(**enc).logits).cpu().numpy()

id2label = model2.config.id2label
for s, p in zip(samples, probs):
    hits = [id2label[i] for i in np.where(p >= THRESHOLD)[0]]
    top = id2label[int(p.argmax())]
    print(f"{s}  →  {hits or '(없음)'}  | 최고 확률: {top} {p.max():.2f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6871.98it/s]

오늘 날씨 진짜 좋다  →  ['not_hate']  | 최고 확률: not_hate 0.99
이 정치인은 정말 최악이다  →  ['not_hate']  | 최고 확률: not_hate 0.60
나이 먹고 왜 저러냐  →  ['age']  | 최고 확률: age 0.76


## 9. 기록 (노션 W3에 붙여넣기)

아래 셀 출력을 그대로 복사해서 노션에 붙이고, 7번 결과표는 캡쳐해서 같이 올리기.

**다음 (W4 v1)**: 설정 셀에서 `SAMPLE_SIZE = None`, `EPOCHS = 3`, `SAVE_DIR`의 `"v0"` → `"v1"`로 바꾸고 그대로 실행 (전체 데이터는 GPU에서 epoch당 몇 분 수준)

In [13]:
log = trainer.state.log_history
train_losses = [x["loss"] for x in log if "loss" in x]
print(f"""- 설정: {MODEL_NAME} / MAX_LENGTH {MAX_LENGTH} / batch {BATCH_SIZE} / lr {LR} / epoch {EPOCHS} / sample {SAMPLE_SIZE} / threshold {THRESHOLD} / class_weight {USE_CLASS_WEIGHT}
- 데이터: train {len(train_ds)} / val {len(val_ds)}
- 학습 시간: {train_result.metrics['train_runtime']:.0f}초 / train loss {(f"{train_losses[0]:.4f} → {train_losses[-1]:.4f}" if train_losses else f"{train_result.training_loss:.4f}")}
- validation: micro F1 {result['micro_f1']:.4f} | macro F1 {result['macro_f1']:.4f}
- 체크포인트 저장: {FINAL_DIR}""")

- 설정: klue/roberta-base / MAX_LENGTH 64 / batch 32 / lr 2e-05 / epoch 3 / sample None / threshold 0.5 / class_weight False
- 데이터: train 78305 / val 8776
- 학습 시간: 321초 / train loss 0.7044 → 0.0640
- validation: micro F1 0.8534 | macro F1 0.7257
- 체크포인트 저장: /home/team17/project/checkpoints/v2/final
